# Computação Quântica na Prática — Nível Introdutório

Este notebook é um workshop prático de introdução à computação quântica, concebido
no formato da *Qiskit Fall Fest*: além de observar resultados, o
objetivo é **compreender por que eles ocorrem** e adquirir a capacidade de
controlar um qubit.

Não é exigido conhecimento prévio de física quântica nem de matemática avançada
(apenas noções elementares de trigonometria, revisadas ao longo do texto). Os
conceitos são construídos de forma incremental:

1. **O que é um qubit** — o vetor de estado, as amplitudes e a regra de Born.
2. **Como visualizar um qubit** — a esfera de Bloch.
3. **Como controlar um qubit** — portas e, sobretudo, rotações (Rx, Ry, Rz).
4. **Medição** — colapso, fase e bases de medição.
5. **Emaranhamento** — o estado de Bell, com dois qubits.
6. **Hardware real** — execução em um computador quântico da IBM e o efeito do ruído.
7. **Aplicação** — geração de números aleatórios.

Ao longo do material há **exercícios avaliados**, nos quais se pede a dedução das
respostas. O corretor indica imediatamente se a resposta está correta ou não e
apresenta a justificativa, sem revelar o valor esperado.

---
**Duração estimada:** cerca de 90 minutos · **Ferramenta:** [Qiskit](https://www.ibm.com/quantum/qiskit)


## 0. Preparação do ambiente

Execute a célula a seguir. Caso o Qiskit ainda não esteja instalado, remova o `#`
da primeira linha antes de executar.

In [ ]:
# %pip install qiskit qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc seaborn

import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere, plot_histogram

import qutils
from grader import verificar, progresso

print("Ambiente carregado com sucesso.")

---
## 1. O que é um qubit? O vetor de estado

Um **bit** clássico assume o valor 0 ou 1. Um **qubit** é descrito por um **vetor
de estado**:

$$|\psi\rangle = \alpha\,|0\rangle + \beta\,|1\rangle$$

- $|0\rangle$ e $|1\rangle$ são os dois estados da base computacional.
- $\alpha$ e $\beta$ são **amplitudes**, números que podem ser complexos.
- Pela **regra de Born**, a probabilidade de cada resultado é o módulo ao quadrado
  da amplitude correspondente: $P(0) = |\alpha|^2$ e $P(1) = |\beta|^2$.
- Como algum resultado necessariamente ocorre, as probabilidades somam 1:
  $|\alpha|^2 + |\beta|^2 = 1$ (o estado é dito **normalizado**).

A célula abaixo exibe o vetor de estado de um qubit recém-inicializado. Todo qubit
começa no estado $|0\rangle$.

In [ ]:
qc = QuantumCircuit(1)                    # 1 qubit, inicializado em |0>
estado = Statevector.from_instruction(qc)
print("Vetor de estado:", estado.data)   # [1+0j, 0+0j]  -> alpha=1, beta=0
print("Probabilidades :", estado.probabilities_dict())
print()
print("Interpretação: amplitude de |0> = 1 -> P(0)=1; amplitude de |1> = 0 -> P(1)=0.")

### Exercício 1 — a regra de Born

Um qubit encontra-se no estado $|\psi\rangle = 0.6\,|0\rangle + 0.8\,|1\rangle$.

Qual é a **probabilidade de medir 1**? Aplique a regra de Born, $P(1) = |\beta|^2$,
e responda com um número decimal entre 0 e 1.

In [ ]:
resposta_1 = None   # substitua pelo valor calculado (ex.: 0.5)

verificar("intro-1", resposta_1)

---
## 2. Visualizando um qubit: a esfera de Bloch

Qualquer estado de **um** qubit pode ser representado como um ponto na superfície
de uma esfera, a **esfera de Bloch** — uma ferramenta central para interpretar o
efeito das portas.

- **Polo norte:** $|0\rangle$
- **Polo sul:** $|1\rangle$
- **Equador:** superposições com probabilidades iguais (50/50)

O ângulo em relação ao eixo vertical (Z) determina as probabilidades de medição; a
posição ao longo do equador corresponde à **fase**, discutida no Módulo 4.

In [ ]:
# O estado |0> localiza-se no polo norte da esfera:
estado_0 = Statevector.from_instruction(QuantumCircuit(1))
plot_bloch_multivector(estado_0)

---
## 3. Controlando o qubit: portas como operações

As portas quânticas transformam o vetor de estado. Três portas fundamentais:

| Porta | Efeito | Na esfera de Bloch |
|---|---|---|
| **X** | Análogo do NOT: troca $|0\rangle \leftrightarrow |1\rangle$ | rotação de 180° em torno de **X** |
| **Z** | Inverte o sinal (fase) de $|1\rangle$ | rotação de 180° em torno de **Z** |
| **H** (Hadamard) | Cria superposição: $|0\rangle \to |+\rangle$ | leva o polo ao equador |

A célula abaixo aplica a Hadamard: o estado deixa o polo norte e passa ao equador,
uma superposição com probabilidades iguais.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)                                   # cria superposição
estado = Statevector.from_instruction(qc)
print("Vetor de estado:", np.round(estado.data, 3))     # [0.707, 0.707]
print("Probabilidades :", estado.probabilities_dict())  # ~50/50
plot_bloch_multivector(estado)

O vetor tornou-se $\tfrac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$: cada amplitude vale
aproximadamente $0.707 \approx \tfrac{1}{\sqrt2}$, e $0.707^2 \approx 0.5$ —
novamente a regra de Born.

### Exercício 2 — a porta X

Complete o circuito de modo a levar o qubit de $|0\rangle$ a $|1\rangle$ com
probabilidade 1 (certeza de medir 1). Identifique a porta apropriada.

O corretor **simula o circuito construído** e verifica a probabilidade resultante.

In [ ]:
meu_circuito = QuantumCircuit(1)

# Insira abaixo a porta que leva |0> a |1> (uma linha):


meu_circuito.measure_all()
verificar("intro-2", meu_circuito)

### Exercício 3 — portas são reversíveis

Uma diferença marcante em relação à computação clássica: **toda porta quântica é
reversível**. Aplicar novamente a mesma porta desfaz o seu efeito e devolve o
qubit ao estado inicial. Essa é a essência do *postulado da evolução* — as
operações quânticas são sempre reversíveis (diferentemente do AND clássico, que
descarta informação).

O circuito abaixo já começa com uma Hadamard, ou seja, no estado $|+\rangle$.
Acrescente **uma** porta que **desfaça** a superposição e retorne o qubit a
$|0\rangle$ — de modo que a probabilidade de medir 1 seja 0.

O corretor simula o circuito e verifica a probabilidade resultante.

In [ ]:
meu_circuito = QuantumCircuit(1)
meu_circuito.h(0)          # o qubit já está em |+> (superposição)

# Acrescente abaixo a porta que retorna o qubit a |0> (uma linha):


meu_circuito.measure_all()
verificar("intro-3", meu_circuito)

---
## 4. Rotações: o mecanismo central de controle

Este é o conceito mais importante do módulo. **Toda** porta de um qubit corresponde
a uma **rotação** na esfera de Bloch. As portas de rotação recebem explicitamente o
**ângulo** $\theta$:

- **Rx($\theta$)** — rotação de $\theta$ radianos em torno do eixo **X**;
- **Ry($\theta$)** — rotação em torno do eixo **Y**;
- **Rz($\theta$)** — rotação em torno do eixo **Z**.

A rotação **Ry** é a mais direta para desenvolver intuição, pois atua diretamente
sobre as probabilidades. Aplicada a $|0\rangle$:

$$\mathrm{Ry}(\theta)\,|0\rangle = \cos\tfrac{\theta}{2}\,|0\rangle + \sin\tfrac{\theta}{2}\,|1\rangle
\quad\Rightarrow\quad P(1) = \sin^2\!\tfrac{\theta}{2}$$

Assim, à medida que $\theta$ varia de 0 a $\pi$, a probabilidade de medir 1 cresce
continuamente de 0 a 1. O gráfico a seguir ilustra essa relação.

In [ ]:
angulos = np.linspace(0, 2*np.pi, 100)
prob_1 = []
for th in angulos:
    c = QuantumCircuit(1)
    c.ry(th, 0)
    prob_1.append(Statevector.from_instruction(c).probabilities()[1])  # P(1)

plt.figure(figsize=(7, 4))
plt.plot(angulos, prob_1, lw=2, label=r"$P(1) = \sin^2(\theta/2)$")
plt.axhline(0.5, color="gray", ls="--", lw=1)
plt.axvline(np.pi, color="gray", ls="--", lw=1)
plt.title("Ry(θ): o ângulo controla a probabilidade de medição")
plt.xlabel("ângulo θ (radianos)"); plt.ylabel("P(medir 1)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

print("θ=0 -> P(1)=0 (estado |0>); θ=π/2 -> P(1)=0.5 (|+>); θ=π -> P(1)=1 (|1>).")

In [ ]:
# Três ângulos representativos na esfera de Bloch: 0, π/2 e π
for th, descr in [(0, "θ=0   -> |0>"), (np.pi/2, "θ=π/2 -> equador"), (np.pi, "θ=π   -> |1>")]:
    c = QuantumCircuit(1); c.ry(th, 0)
    print(descr, "->", np.round(Statevector.from_instruction(c).probabilities(), 3))
c = QuantumCircuit(1); c.ry(np.pi/2, 0)
plot_bloch_multivector(Statevector.from_instruction(c))

### Exercício 4 — dedução do ângulo

Utilize uma rotação **Ry** para preparar um estado com **25% de probabilidade** de
medir 1, isto é, $P(1) = 0.25$.

É necessário **deduzir o ângulo $\theta$**. A partir de $P(1) = \sin^2(\theta/2)$,
iguale a 0.25 e resolva para $\theta$.

Sugestão: as funções `np.pi`, `np.sqrt` e `np.arcsin` podem auxiliar; o ângulo é
expresso em radianos.

In [ ]:
theta = 0.0          # calcule e ajuste o ângulo

meu_circuito = QuantumCircuit(1)
meu_circuito.ry(theta, 0)
meu_circuito.measure_all()

# Verificação prévia, feita por você:
print("P(1) do seu circuito =", Statevector.from_instruction(
      meu_circuito.remove_final_measurements(inplace=False)).probabilities()[1])

verificar("intro-4", meu_circuito)

### Exercício 5 — o mesmo controle, outra probabilidade

Reforce o raciocínio anterior no sentido oposto: use uma rotação **Ry** para
preparar um estado com **75% de probabilidade** de medir 1, isto é,
$P(1) = 0.75$.

Deduza o ângulo a partir de $P(1) = \sin^2(\theta/2)$. Ao terminar, compare o
valor com o ângulo do Exercício 4 — há uma simetria interessante entre eles.

In [ ]:
theta = 0.0          # calcule e ajuste o ângulo

meu_circuito = QuantumCircuit(1)
meu_circuito.ry(theta, 0)
meu_circuito.measure_all()

# Verificação prévia, feita por você:
print("P(1) do seu circuito =", Statevector.from_instruction(
      meu_circuito.remove_final_measurements(inplace=False)).probabilities()[1])

verificar("intro-5", meu_circuito)

### A fase e a porta Rz

A porta `Rz` executa uma rotação em torno do eixo **Z** (ao longo do equador). Ela
altera a **fase relativa** entre $|0\rangle$ e $|1\rangle$, mas **não** modifica as
probabilidades de medição na base computacional. Na célula abaixo, `Rz` é aplicada
sobre $|+\rangle$ e as probabilidades permanecem 50/50, embora as amplitudes
adquiram um componente complexo.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)                 # prepara |+>
qc.rz(np.pi/2, 0)       # rotaciona a fase
estado = Statevector.from_instruction(qc)
print("Amplitudes (observe o componente complexo):", np.round(estado.data, 3))
print("Probabilidades em Z (inalteradas)          :", estado.probabilities_dict())

### Exercício 6 — o efeito de Rz

A aplicação de `Rz(θ)` a um qubit altera principalmente o quê?

- **(a)** As probabilidades de medir 0 ou 1 na base computacional.
- **(b)** A fase relativa entre $|0\rangle$ e $|1\rangle$, sem alterar as probabilidades em Z.
- **(c)** Converte o qubit em um bit clássico.
- **(d)** Nada — `Rz` não produz efeito.

In [ ]:
resposta_6 = ""   # "a", "b", "c" ou "d"

verificar("intro-6", resposta_6)

---
## 5. Medição, fase e bases de medição

A medição difere das operações anteriores: ela provoca o **colapso** da
superposição em um valor definido (0 ou 1), com probabilidades dadas pela regra de
Born. Como cada medição é aleatória, o procedimento é repetido muitas vezes
(*shots*) para estimar as probabilidades.

Há aqui um ponto conceitual importante. Os estados
$|+\rangle = \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle)$ e
$|-\rangle = \tfrac{1}{\sqrt2}(|0\rangle - |1\rangle)$ possuem **as mesmas
probabilidades** (50/50) quando medidos na base computacional (Z): diferem apenas
no sinal (fase) da amplitude de $|1\rangle$.

Se a medição em Z não os distingue, é necessário mudar a **base de medição**.
Aplicando uma Hadamard **antes** da medição (o que equivale a medir na base X),
obtém-se $|+\rangle \to |0\rangle$ e $|-\rangle \to |1\rangle$.

In [ ]:
def medir(circ, base="Z", shots=2000):
    c = circ.copy()
    if base == "X":
        c.h(0)                # medir na base X equivale a aplicar H antes da medição
    c.measure_all()
    return qutils.probabilidades(qutils.simular(c, shots=shots))

mais  = QuantumCircuit(1); mais.h(0)               # |+>
menos = QuantumCircuit(1); menos.x(0); menos.h(0)  # |->

print("Base Z (não distingue os estados):")
print("   |+> ->", medir(mais,  "Z"))
print("   |-> ->", medir(menos, "Z"))
print("Base X (distingue os estados):")
print("   |+> ->", medir(mais,  "X"), "-> resulta em 0")
print("   |-> ->", medir(menos, "X"), "-> resulta em 1")

### Exercício 7 — por que a fase não aparece na base Z

$|+\rangle$ e $|-\rangle$ produzem o mesmo resultado (50/50) ao serem medidos na
base computacional (Z). A que se deve isso e como distingui-los?

- **(a)** São o mesmo estado; não há como distingui-los.
- **(b)** Diferem em probabilidade; basta aumentar o número de medições em Z.
- **(c)** Diferem apenas na fase, que a medição em Z não capta; para distingui-los,
  mede-se na base X (aplicando H antes da medição).
- **(d)** Somente um computador quântico real (com ruído) consegue distingui-los.

In [ ]:
resposta_7 = ""   # "a", "b", "c" ou "d"

verificar("intro-7", resposta_7)

### Exercício 8 — escolhendo a base de medição

Ponha em prática a ideia acima. Prepare o estado
$|-\rangle = \tfrac{1}{\sqrt2}(|0\rangle - |1\rangle)$ e **meça-o na base X**,
usando a função `medir` definida mais acima.

Execute a célula e observe a saída: na base X, o estado $|-\rangle$ produz sempre
um resultado definido. **Informe qual bit aparece** (`"0"` ou `"1"`). A resposta é
o próprio resultado da execução.

In [ ]:
menos = QuantumCircuit(1)

# Prepare |-> a partir de |0> (dica: X seguido de H):


resultado = medir(menos, base="X")     # base X: aplica H antes de medir
print("Resultado na base X:", resultado)

resposta_8 = ""      # digite o bit observado: "0" ou "1"
verificar("intro-8", resposta_8)

---
## 6. Dois qubits: produto tensorial e emaranhamento

Com dois qubits, o vetor de estado possui **quatro** componentes — as amplitudes de
$|00\rangle, |01\rangle, |10\rangle, |11\rangle$. Com $N$ qubits são $2^N$
componentes: esse crescimento exponencial está na origem do potencial da computação
quântica.

Quando os qubits são independentes, o estado é o produto dos estados individuais.
Existem, porém, estados que **não** admitem essa decomposição — os estados
**emaranhados**. O exemplo canônico é o **estado de Bell**, construído com duas
portas:

1. **Hadamard no qubit 0**, criando superposição;
2. **CNOT (`cx`) com controle 0 e alvo 1**. O CNOT inverte o alvo somente quando o
   controle vale 1, o que correlaciona os dois qubits.

### Exercício 9 — construção do estado de Bell

In [ ]:
meu_bell = QuantumCircuit(2)

# Insira abaixo as duas portas:
# 1) Hadamard no qubit 0
# 2) CNOT com controle 0 e alvo 1   ->   .cx(0, 1)


verificar("intro-9", meu_bell)

A versão de referência é utilizada a seguir. O estado é visualizado com a
*q-sphere*, que representa as amplitudes e fases de sistemas de múltiplos qubits.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
estado = Statevector.from_instruction(bell)
print("Vetor de estado (4 amplitudes):", np.round(estado.data, 3))
print("Somente |00> e |11> possuem amplitude não nula.")
plot_state_qsphere(estado)

In [ ]:
bell_medido = bell.copy(); bell_medido.measure_all()
contagens = qutils.simular(bell_medido, shots=2000)
print("Medições:", qutils.probabilidades(contagens))
plot_histogram(contagens, title="Estado de Bell: apenas 00 e 11")

Apenas `00` e `11` ocorrem; `01` e `10` nunca aparecem. Cada medição individual é
aleatória, mas os dois qubits sempre concordam — a manifestação do emaranhamento.

### Exercício 10 — resultados do emaranhamento

No estado de Bell, quais pares de bits podem ser observados na medição? Responda
como um conjunto, por exemplo `{"00", "11"}`.

In [ ]:
resposta_10 = set()   # ex.: {"00", "11"}

verificar("intro-10", resposta_10)

### Exercício 11 — um par anti-correlacionado

No estado de Bell os qubits saem sempre **iguais** (`{00, 11}`). Modifique o
circuito para que passem a sair sempre **opostos** — um par *anti-correlacionado*.

Partindo do estado de Bell (H no qubit 0, depois CNOT), acrescente **uma porta X**
em um dos qubits. Execute a célula, observe as medições e **informe o conjunto de
resultados possíveis** (por exemplo, `{"00", "11"}`). A resposta é o que você
observa na saída.

In [ ]:
anti = QuantumCircuit(2)
anti.h(0)
anti.cx(0, 1)

# Acrescente abaixo uma porta X para inverter um dos qubits:


anti.measure_all()
contagens = qutils.simular(anti, shots=2000)
print("Medições:", qutils.probabilidades(contagens))

resposta_11 = set()      # ex.: {"00", "11"}
verificar("intro-11", resposta_11)

---
## 7. Execução em um computador quântico real (IBM Quantum)

Até este ponto foi utilizado o **simulador**, que reproduz a matemática ideal, sem
erros. Trata-se de um recurso adequado para o aprendizado, mas a simulação torna-se
exponencialmente custosa: por volta de 50 qubits já é inviável. Por essa razão
existe o hardware real. Nesta seção, o estado de Bell é enviado a um computador
quântico da IBM.

### Obtenção de acesso gratuito (procedimento único)
1. Crie uma conta gratuita na **IBM Quantum Platform**: https://quantum.cloud.ibm.com/
   (o plano **Open** é gratuito e oferece cerca de 10 minutos de processamento
   quântico a cada 28 dias).
2. Gere uma **API key** e copie o **CRN** da respectiva *instance*.
3. Informe essas credenciais na célula abaixo, ou salve a conta em disco (ver o
   arquivo `README.md`).

O canal a ser utilizado é `ibm_quantum_platform`; o canal anterior, `ibm_quantum`,
foi descontinuado em 1 de julho de 2025. Por segurança, não compartilhe seu token.

In [ ]:
# Deixe TOKEN_IBM em branco para ignorar o hardware real. Nesse caso, utiliza-se um
# simulador COM RUÍDO, que reproduz o comportamento de um processador físico.
TOKEN_IBM = ""        # API key da IBM Quantum
INSTANCE_IBM = ""     # CRN da instance (se necessário)

if TOKEN_IBM:
    print("Conectando à IBM Quantum...")
    service = qutils.conectar_ibm(TOKEN_IBM, INSTANCE_IBM or None)
    contagens_real, nome_backend, job_id = qutils.rodar_no_hardware(
        bell_medido, service, shots=1024)
    fonte = "computador real: " + nome_backend
else:
    print("Sem token — utilizando um simulador com ruído para reproduzir o hardware real.")
    contagens_real = qutils.simular_com_ruido(bell_medido, shots=1024)
    fonte = "simulador com ruído (reproduz hardware real)"

print("Fonte dos resultados:", fonte)
print("Contagens:", contagens_real)

In [ ]:
contagens_ideal = qutils.simular(bell_medido, shots=1024)
print("Ideal (simulador):", qutils.probabilidades(contagens_ideal))
print("Real / com ruído :", qutils.probabilidades(contagens_real))
plot_histogram([contagens_ideal, contagens_real],
               title="Simulador ideal versus computador real (com ruído)",
               legend=["ideal", "real/ruído"])

### O efeito do ruído

No caso ideal, apenas `00` e `11` são observados. No hardware real (ou no simulador
com ruído) surgem também ocorrências de `01` e `10` — resultados proibidos pelo
estado ideal. O hardware é físico e está sujeito a vibração, temperatura,
imperfeições nas portas e **decoerência** (a perda gradual do estado quântico).
Esse conjunto de efeitos constitui o **ruído**, cuja redução é o principal desafio
técnico da área.

### Exercício 12 — a diferença entre hardware real e simulador

- **(a)** O simulador está incorreto.
- **(b)** O ruído e a decoerência do hardware físico.
- **(c)** O computador real utiliza bits, não qubits.
- **(d)** A mecânica quântica varia de um dia para o outro.

In [ ]:
resposta_12 = ""   # "a", "b", "c" ou "d"

verificar("intro-12", resposta_12)

---
## 8. Uma aplicação: geração de números aleatórios

Computadores clássicos produzem números **pseudoaleatórios**, gerados por fórmulas
determinísticas. Um computador quântico gera aleatoriedade **genuína**, de origem
física: basta colocar qubits em superposição com Hadamard e medi-los. Cada qubit
fornece um bit efetivamente aleatório — recurso empregado, por exemplo, em
criptografia.

In [ ]:
def numero_aleatorio_quantico(n_bits=4):
    qc = QuantumCircuit(n_bits)
    qc.h(range(n_bits))          # todos os qubits em superposição
    qc.measure_all()
    contagens = qutils.simular(qc, shots=1)      # 1 shot = 1 amostra
    bits = next(iter(contagens))
    return int(bits, 2), bits

valor, bits = numero_aleatorio_quantico(4)
print("Bits gerados:", bits, "->", valor, "(decimal)")
print("A cada execução o valor muda, de forma genuinamente aleatória.")

### Exercício 13 — espaço de resultados

Um gerador quântico baseado em **4 qubits** produz quantos valores distintos
(de `0000` a `1111`)?

In [ ]:
resposta_13 = 0   # informe o número

verificar("intro-13", resposta_13)

---
## 9. Síntese

Sem recorrer ao jargão, este workshop percorreu os **quatro postulados** que
fundamentam a mecânica quântica:

1. **O estado** — um qubit é um vetor de amplitudes (Módulo 1).
2. **A evolução** — portas *reversíveis* transformam esse vetor (Módulos 3 e 4).
3. **A medição** — o resultado é probabilístico e depende da *base* escolhida (Módulos 1 e 5).
4. **Sistemas compostos** — vários qubits ocupam um espaço muito maior, no qual
   surge o emaranhamento (Módulo 6).

| Objetivo | Atividade realizada |
|---|---|
| **O que é um qubit** | Leitura do vetor de estado, aplicação da regra de Born e distinção entre amplitude e fase. |
| **Como controlar** | Uso de portas e rotações (X, H, Ry, Rz), verificação de que toda porta é reversível e dedução de ângulos para atingir probabilidades dadas. |
| **Como medir** | Compreensão do colapso, dos *shots* e do papel da base de medição. |
| **Emaranhamento** | Construção e visualização de um estado de Bell na *q-sphere* e criação de um par anti-correlacionado. |
| **Hardware real** | Execução em processador físico e quantificação do efeito do ruído. |
| **Aplicação** | Geração de números aleatórios de origem quântica. |

### Sugestões de continuidade
- Modifique parâmetros (`theta`, número de *shots*), combine portas e experimente
  outras bases de medição.
- Prossiga para o notebook **intermediário** (estados GHZ, interferência, algoritmos
  de Deutsch–Jozsa e de Bernstein–Vazirani, busca de Grover).
- Consulte o [Qiskit Learning](https://learning.quantum.ibm.com/) da IBM.

Execute a célula abaixo para visualizar o progresso nos exercícios.

In [ ]:
progresso("intro")